# Phase 8: generation runs on Colab

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

Run the cells **in order**, one at a time (click ▶ or press Shift+Enter). Wait for the green ✓ before the next one.
Needs `rag_data.zip` (data/raw + data/processed) in the top level of your Google Drive.

**If the runtime disconnects:** start again from cell 1. Answers are saved to `My Drive/gen_llama` as they are made, so cell 4 continues where it stopped.

## 1. Connect Drive, get the code and data, install packages (2-4 min)
A pop-up asks for Drive access: click **Connect to Google Drive → Allow**. Red pip 'dependency conflict' warnings are normal.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone -b phase-8-generation https://github.com/himalSubedi0602/Rag_System.git /content/rag
%cd /content/rag
!unzip -q -o /content/drive/MyDrive/rag_data.zip -d /content/rag
!pip install -q -r requirements.txt

## 2. Install Ollama and download both models (~5 min)
Each `ollama pull` ends with `success`, and the last line lists **both** models.

In [ ]:
!sudo apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(10)
!ollama pull llama3.1:8b
!ollama pull qwen2.5:7b
!ollama list

## 3. Tests and a 5-question trial (~2 min)
Uses the GPU for embeddings (`device: auto`; the CPU setting is only for the 8 GB Mac). Expect `138 passed`, then 5 progress lines and a small table.

In [ ]:
%cd /content/rag
!sed -i 's/^device: cpu/device: auto/' configs/gen_llama.yaml
!grep device configs/gen_llama.yaml
!python -m pytest -q
!python run_generation.py configs/gen_llama.yaml --limit 5

## 4. Full tune run, saved to Drive as it goes
About 159 questions (~15 min) plus grading. Keep this tab open and in front. Results are written straight to `My Drive/gen_llama`;
if the runtime disconnects, run cells 1-2, then this cell again: it resumes from the saved answers.

In [ ]:
%cd /content/rag
!sed -i 's/^device: cpu/device: auto/' configs/gen_llama.yaml
!mkdir -p /content/drive/MyDrive/gen_llama
!rm -rf eval/results/gen_llama && ln -s /content/drive/MyDrive/gen_llama eval/results/gen_llama
!python run_generation.py configs/gen_llama.yaml
!ls /content/drive/MyDrive/gen_llama

## 5. Fallback model, only if the trigger fires (pre-registered)
Reads gen_llama's citation scores. If format < 0.90 or support < 0.70, it runs the same questions with `qwen2.5:7b`
(saved to `My Drive/gen_qwen`, resumable like cell 4). Otherwise it prints that no fallback is needed and does nothing.

In [ ]:
%cd /content/rag
import pandas as pd, subprocess
m = pd.read_csv('eval/results/gen_llama/metrics.csv', index_col=0).loc['overall']
fmt, sup = m['cit_format'], m['cit_supported']
print(f'gen_llama overall: citation format {fmt:.3f} (trigger < 0.90), support {sup:.3f} (trigger < 0.70)')
if fmt < 0.90 or sup < 0.70:
    print('Trigger fired: running the qwen2.5:7b fallback.')
    !sed -i 's/^device: cpu/device: auto/' configs/gen_qwen.yaml
    !mkdir -p /content/drive/MyDrive/gen_qwen
    !rm -rf eval/results/gen_qwen && ln -s /content/drive/MyDrive/gen_qwen eval/results/gen_qwen
    !python run_generation.py configs/gen_qwen.yaml
else:
    print('No fallback needed: llama3.1:8b stays. You are done with Colab for now.')